# RNN Text Classification with Different Embeddings

This notebook implements **RNN-based text classification** using:
1. Trainable (normal) word embeddings
2. Word2Vec Skip-gram
3. Word2Vec CBOW


## 1. Setup & Imports

In [1]:
print("Installing required packages...")

!pip install -U pip setuptools wheel Cython
!pip install gensim>=4.3.2
!pip install tensorflow>=2.10.0
!pip install scikit-learn>=1.0.0
!pip install matplotlib>=3.4.0
!pip install seaborn>=0.11.0
!pip install numpy>=1.21.0
!pip install pandas>=1.3.0

print("Installation complete.")

Installing required packages...
  Using cached setuptools-81.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached wheel-0.46.3-py3-none-any.whl.metadata (2.4 kB)
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ----------- ---------------------------- 0.5/1.8 MB 2.2 MB/s eta 0:00:01
   ----------------------- ---------------- 1.0/1.8 MB 2.8 MB/s eta 0:00:01
   ----------------------------------- ---- 1.6/1.8 MB 2.4 MB/s eta 0:00:01
   ---------------------------------------- 1.8/1.8 MB 2.4 MB/s eta 0:00:00
Using cached setuptools-81.0.0-py3-none-any.whl (1.1 MB)
Using cached wheel-0.46.3-py3-none-any.whl (30 kB)
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   --- ------------------------------------ 0.3/2.8 MB ? eta -:--:--
   ----------- ---------------------------- 0.8/2.8 MB 2.2 MB/s eta 0:00:01
   ------------------- -------------------- 1.3/2.8 MB 2.7 MB/s eta 0:00:


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: To modify pip, please run the following command:
C:\Users\ADVANCED TECH\AppData\Local\Programs\Python\Python313\python.exe -m pip install -U pip setuptools wheel Cython

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 26.0.1


Installation complete.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.preprocessing import LabelEncoder
from sklearn.datasets import fetch_20newsgroups

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Dropout, Embedding, Bidirectional, SpatialDropout1D
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


np.random.seed(42)
tf.random.set_seed(42)



## 2. Load Dataset 

In [6]:

categories = [
    'comp.graphics',
    'sci.med',
    'rec.sport.baseball',
    'talk.politics.misc'
]

train_data = fetch_20newsgroups(
    subset='train',
    categories=categories,
    remove=('headers', 'footers', 'quotes'),
    random_state=42
)

test_data = fetch_20newsgroups(
    subset='test',
    categories=categories,
    remove=('headers', 'footers', 'quotes'),
    random_state=42
)

texts = list(train_data.data) + list(test_data.data)
labels = [train_data.target_names[i] for i in train_data.target] +          [test_data.target_names[i] for i in test_data.target]

print("Documents:", len(texts))
print("Classes:", categories)


Documents: 3732
Classes: ['comp.graphics', 'sci.med', 'rec.sport.baseball', 'talk.politics.misc']


## 3. Text Preprocessing (matched)

In [7]:

def preprocess_text(texts):
    processed = []
    for text in texts:
        text = text.lower()
        text = ' '.join(text.split())
        processed.append(text)
    return processed

texts = preprocess_text(texts)

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(labels)

print("Encoded classes:", label_encoder.classes_)


Encoded classes: ['comp.graphics' 'rec.sport.baseball' 'sci.med' 'talk.politics.misc']


## 4. Tokenization & Padding

In [8]:

MAX_WORDS = 10000
MAX_LEN = 100

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(texts)

sequences = tokenizer.texts_to_sequences(texts)
X = pad_sequences(sequences, maxlen=MAX_LEN, padding='post', truncating='post')

print("Vocabulary size:", len(tokenizer.word_index))
print("Input shape:", X.shape)


Vocabulary size: 36779
Input shape: (3732, 100)


## 5. Train-Test Split

In [9]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)


Train: (2985, 100)
Test: (747, 100)


## 6. Model Builder (RNN)

In [13]:

def build_rnn_model(embedding_matrix=None, embedding_dim=100, rnn_units=128, num_classes=4):
    model = Sequential()

    if embedding_matrix is not None:
        model.add(Embedding(
            input_dim=embedding_matrix.shape[0],
            output_dim=embedding_dim,
            weights=[embedding_matrix],
            input_length=MAX_LEN,
            trainable=False
        ))
    else:
        model.add(Embedding(
            input_dim=min(len(tokenizer.word_index) + 1, MAX_WORDS),
            output_dim=embedding_dim,
            input_length=MAX_LEN
        ))

    model.add(SpatialDropout1D(0.2))
    model.add(Bidirectional(SimpleRNN(rnn_units)))
    model.add(Dropout(0.3))
    model.add(Dense(num_classes, activation='softmax'))

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


## 7. Experiment 1: Trainable Embeddings

In [14]:

model_trainable = build_rnn_model(num_classes=len(label_encoder.classes_))
model_trainable.summary()

history_trainable = model_trainable.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=64,
    callbacks=[EarlyStopping(patience=3, restore_best_weights=True)],
    verbose=1
)


c:\Users\ADVANCED TECH\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 13s 131ms/step - accuracy: 0.2831 - loss: 1.4094 - val_accuracy: 0.3183 - val_loss: 1.4041
Epoch 2/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 4s 112ms/step - accuracy: 0.3794 - loss: 1.3185 - val_accuracy: 0.3116 - val_loss: 1.3741
Epoch 3/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 6s 119ms/step - accuracy: 0.5637 - loss: 1.1211 - val_accuracy: 0.3501 - val_loss: 1.3544
Epoch 4/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 5s 117ms/step - accuracy: 0.7751 - loss: 0.7856 - val_accuracy: 0.3400 - val_loss: 1.4282
Epoch 5/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 5s 125ms/step - accuracy: 0.8300 - loss: 0.5958 - val_accuracy: 0.3534 - val_loss: 1.5699
Epoch 6/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 5s 120ms/step - accuracy: 0.8957 - loss: 0.4040 - val_accuracy: 0.3585 - val_loss: 1.5409


## 8. Experiment 2: Word2Vec Skip-gram

In [15]:

tokenized_texts = [text.split() for text in texts]

w2v_skip = Word2Vec(
    sentences=tokenized_texts,
    vector_size=100,
    window=5,
    min_count=1,
    sg=1,
    epochs=10
)

vocab_size = min(len(tokenizer.word_index) + 1, MAX_WORDS)
embedding_matrix_skip = np.zeros((vocab_size, 100))

for word, idx in tokenizer.word_index.items():
    if idx < MAX_WORDS and word in w2v_skip.wv:
        embedding_matrix_skip[idx] = w2v_skip.wv[word]

model_skip = build_rnn_model(
    embedding_matrix=embedding_matrix_skip,
    num_classes=len(label_encoder.classes_)
)

history_skip = model_skip.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=64,
    callbacks=[EarlyStopping(patience=3, restore_best_weights=True)],
    verbose=1
)


Epoch 1/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 10s 108ms/step - accuracy: 0.3472 - loss: 1.4049 - val_accuracy: 0.4690 - val_loss: 1.1854
Epoch 2/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 4s 77ms/step - accuracy: 0.5000 - loss: 1.1632 - val_accuracy: 0.5745 - val_loss: 1.0304
Epoch 3/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 6s 92ms/step - accuracy: 0.5611 - loss: 1.0264 - val_accuracy: 0.5544 - val_loss: 1.0784
Epoch 4/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 4s 95ms/step - accuracy: 0.5172 - loss: 1.1781 - val_accuracy: 0.4891 - val_loss: 1.1840
Epoch 5/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 5s 93ms/step - accuracy: 0.5704 - loss: 1.0121 - val_accuracy: 0.6198 - val_loss: 0.9377
Epoch 6/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 4s 93ms/step - accuracy: 0.6202 - loss: 0.9766 - val_accuracy: 0.6147 - val_loss: 0.9076
Epoch 7/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 5s 91ms/step - accuracy: 0.6591 - loss: 0.8568 - val_accuracy: 0.6466 - val_loss: 0.9025
Epoch 8/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 5s 89ms/step - accuracy: 0.6667 - loss: 0.8428 - val_accuracy: 0.6683 -

## 9. Experiment 3: Word2Vec CBOW

In [16]:

w2v_cbow = Word2Vec(
    sentences=tokenized_texts,
    vector_size=100,
    window=5,
    min_count=1,
    sg=0,
    epochs=10
)

embedding_matrix_cbow = np.zeros((vocab_size, 100))

for word, idx in tokenizer.word_index.items():
    if idx < MAX_WORDS and word in w2v_cbow.wv:
        embedding_matrix_cbow[idx] = w2v_cbow.wv[word]

model_cbow = build_rnn_model(
    embedding_matrix=embedding_matrix_cbow,
    num_classes=len(label_encoder.classes_)
)

history_cbow = model_cbow.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=64,
    callbacks=[EarlyStopping(patience=3, restore_best_weights=True)],
    verbose=1
)


Epoch 1/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 10s 102ms/step - accuracy: 0.2952 - loss: 1.5637 - val_accuracy: 0.3853 - val_loss: 1.3354
Epoch 2/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 3s 76ms/step - accuracy: 0.3861 - loss: 1.3487 - val_accuracy: 0.4137 - val_loss: 1.2648
Epoch 3/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 6s 84ms/step - accuracy: 0.4070 - loss: 1.2889 - val_accuracy: 0.4489 - val_loss: 1.2325
Epoch 4/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 5s 81ms/step - accuracy: 0.4485 - loss: 1.2506 - val_accuracy: 0.4020 - val_loss: 1.2609
Epoch 5/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 3s 89ms/step - accuracy: 0.4564 - loss: 1.2176 - val_accuracy: 0.4104 - val_loss: 1.2556
Epoch 6/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 5s 86ms/step - accuracy: 0.4678 - loss: 1.2034 - val_accuracy: 0.4255 - val_loss: 1.2440


## 10. Final Evaluation

In [17]:

def evaluate(model, name):
    y_pred = np.argmax(model.predict(X_test), axis=1)
    print(f"\n{name}")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

evaluate(model_trainable, "RNN – Trainable Embedding")
evaluate(model_skip, "RNN – Word2Vec Skip-gram")
evaluate(model_cbow, "RNN – Word2Vec CBOW")


24/24 ━━━━━━━━━━━━━━━━━━━━ 4s 148ms/step

RNN – Trainable Embedding
Accuracy: 0.3493975903614458
                    precision    recall  f1-score   support

     comp.graphics       0.38      0.31      0.34       195
rec.sport.baseball       0.34      0.48      0.40       199
           sci.med       0.35      0.35      0.35       198
talk.politics.misc       0.33      0.23      0.27       155

          accuracy                           0.35       747
         macro avg       0.35      0.34      0.34       747
      weighted avg       0.35      0.35      0.34       747

24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step

RNN – Word2Vec Skip-gram
Accuracy: 0.6586345381526104
                    precision    recall  f1-score   support

     comp.graphics       0.84      0.78      0.81       195
rec.sport.baseball       0.76      0.80      0.78       199
           sci.med       0.52      0.54      0.53       198
talk.politics.misc       0.49      0.48      0.49       155

          accuracy     

## 11.References

**References**

Elman, J. L. (1990). Finding structure in time. Cognitive Science, 14(2), 179–211.

Hochreiter, S., & Schmidhuber, J. (1997). Long short-term memory. Neural Computation, 9(8), 1735–1780.

Mikolov, T., Chen, K., Corrado, G., & Dean, J. (2013). Efficient estimation of word representations in vector space. arXiv preprint arXiv:1301.3781.

Mikolov, T., Sutskever, I., Chen, K., Corrado, G., & Dean, J. (2013). Distributed representations of words and phrases and their compositionality. Advances in Neural Information Processing Systems, 26.

Kim, Y. (2014). Convolutional neural networks for sentence classification. arXiv preprint arXiv:1408.5882.

Pennington, J., Socher, R., & Manning, C. D. (2014). GloVe: Global vectors for word representation. In Proceedings of the 2014 Conference on Empirical Methods in Natural Language Processing (EMNLP) (pp. 1532–1543).